In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("IMDB Dataset.csv")
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
df.shape

(50000, 2)

In [4]:
df.isnull().sum()

review       0
sentiment    0
dtype: int64

In [5]:
df.drop_duplicates(inplace=True)

In [6]:
df.shape

(49582, 2)

# Text Preprocessing

### 1. Converting to Lowercase

In [7]:
df['review'] = df['review'].str.lower()

### 2. Removing the URLs

In [8]:
import re

In [9]:
def remove_urls(text):
    text = re.sub(r"http\S+", "", text)        # (pattern, repl, string)  eg- https://www.google.com
    return text

df['review'] = df['review'].apply(remove_urls)

### 3. Removing Punctuations

In [10]:
def remove_punctuation(text):
    text = re.sub(r"[^A-Za-z0-9\s]", "", text)        # (pattern, repl, string)  eg- https://www.google.com
    return text

df['review'] = df['review'].apply(remove_punctuation)

### 4. Removing HTML

In [11]:
def remove_html(text):
    text = re.sub(r"<.*?>", "", text)
    return text

df['review'] = df['review'].apply(remove_html)

### 5. Removing the Stopwords

In [12]:
import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [14]:
def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    for word in tokens :
        if word in stop_words:
            text.replace(word, "")
            
    return text

df['review'] = df['review'].apply(remove_stopwords)

### 6. Stemming

In [15]:
from nltk.stem import PorterStemmer

In [16]:
def stemming(text):
    ps = PorterStemmer()
    tokens = word_tokenize(text)
    stemmed_words = []
    
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

df['review'] = df['review'].apply(stemming)

In [17]:
df

,review,sentiment
0,one of the other review ha mention that after ...,positive
1,a wonder littl product br br the film techniqu...,positive
2,i thought thi wa a wonder way to spend time on...,positive
3,basic there a famili where a littl boy jake th...,negative
4,petter mattei love in the time of money is a v...,positive
...,...,...
49995,i thought thi movi did a down right good job i...,positive
49996,bad plot bad dialogu bad act idiot direct the ...,negative
49997,i am a cathol taught in parochi elementari sch...,negative
49998,im go to have to disagre with the previou comm...,negative


### 7. Encoding

In [18]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["sentiment"] = le.fit_transform(df["sentiment"])

In [19]:
y = df["sentiment"]

In [20]:
df

,review,sentiment
0,one of the other review ha mention that after ...,1
1,a wonder littl product br br the film techniqu...,1
2,i thought thi wa a wonder way to spend time on...,1
3,basic there a famili where a littl boy jake th...,0
4,petter mattei love in the time of money is a v...,1
...,...,...
49995,i thought thi movi did a down right good job i...,1
49996,bad plot bad dialogu bad act idiot direct the ...,0
49997,i am a cathol taught in parochi elementari sch...,0
49998,im go to have to disagre with the previou comm...,0


### 8. Vectorization

In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer
tf = TfidfVectorizer(max_features=5000)
X = tf.fit_transform(df["review"])
# print(X)

In [22]:
print(type(X))

<class 'scipy.sparse._csr.csr_matrix'>


In [23]:
X = X.toarray()

# Train-Test-Split

In [24]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [25]:
X_train.shape

(39665, 5000)

In [26]:
X_test.shape

(9917, 5000)

# Dataset & Data Loaders

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [28]:
# X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
# y_train_tensor = torch.tensor(y_train.values, dtype=torch.int32)

# X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
# y_test_tensor = torch.tensor(y_test.values, dtype=torch.int32)

# train_set = TensorDataset(X_train_tensor, y_train_tensor)
# test_set = TensorDataset(X_test_tensor, y_test_tensor)


# ================================ OR ==================================

train_set = TensorDataset(torch.from_numpy(X_train).float(), torch.from_numpy(y_train.values).float())
test_set = TensorDataset(torch.from_numpy(X_test).float(), torch.from_numpy(y_test.values).float())

In [29]:
train_loader = DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64)

# Build our RNN

In [30]:
class RNN(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=1):
        super().__init__()
        
        self.num_layers = num_layers
        self.hidden_size = hidden_size

        # 1 hidden layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):

        # optional => shape(num_layers, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)
        
        out, _ = self.rnn(x, h0)
        # 1st value => hidden state of all the timesteps
        # 2nd value => final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

# Training and Evaluation

In [31]:
input_size = X_train.shape[1]

model = RNN(input_size)

criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters())

In [32]:
epochs = 10

for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        
        optimizer.zero_grad()
        
        Xb = Xb.unsqueeze(1) # Add singleton direction
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size, ) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backward propagation
        optimizer.step()

    print(f"epoch {epoch+1}/{epochs} and loss = {loss.item()}")

epoch 1/10 and loss = 0.24394166469573975
epoch 2/10 and loss = 0.20272737741470337
epoch 3/10 and loss = 0.19662031531333923
epoch 4/10 and loss = 0.18913140892982483
epoch 5/10 and loss = 0.20079059898853302
epoch 6/10 and loss = 0.17361384630203247
epoch 7/10 and loss = 0.2580282986164093
epoch 8/10 and loss = 0.23411142826080322
epoch 9/10 and loss = 0.1556142270565033
epoch 10/10 and loss = 0.2705957591533661


In [33]:
model.eval()

with torch.no_grad():
    correct_vals = 0
    total_vals = 0
    
    for Xb, yb in test_loader:   
        Xb = Xb.unsqueeze(1)
        
        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        total_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/total_vals*100}")

accuracy = 87.4760512251689


In [34]:
import joblib

# Save your trained TF-IDF vectorizer
joblib.dump(tf, 'tfidf_vectorizer.joblib')

# Save your trained PyTorch model weights
torch.save(model.state_dict(), 'rnn_model.pth')